# 04 – Klassifikation: Vorhersage von TARGET

**CRISP-DM-Phase:** Modeling & Evaluation

In diesem Notebook trainieren wir fünf Klassifikationsmodelle zur Vorhersage von Kreditausfällen (`TARGET = 1`) und vergleichen sie systematisch. Der Datensatz ist stark **unausgewogen** (~8 % Ausfälle), was besondere Anforderungen an Metriken und die Wahl des Entscheidungsschwellenwerts stellt.

**Ablauf:**
1. Stratifizierter Train-Test-Split
2. Modellvergleich per Cross-Validation (PR-AUC)
3. Finales Fitten und Evaluation auf dem Testset
4. ROC- und Precision-Recall-Kurven
5. Schwellenwert-Analyse (Precision/Recall-Trade-off)
6. Confusion Matrix und Classification Report
7. Kalibrierung
8. Hyperparameter-Tuning und Speichern des finalen Modells

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, utils, preprocessing as prep, train_models as tm, evaluate_models as ev
feat = utils.load_processed("feature_matrix")
print("Feature-Matrix:", feat.shape)

Feature-Matrix: (307511, 161)


## 1. Train-Test-Split (stratifiziert)

Der Datensatz wird im Verhältnis **80/20** in Trainings- und Testdaten aufgeteilt. Die Aufteilung ist **stratifiziert**: Der Anteil der Kreditausfälle (~8 %) bleibt in beiden Teilmengen gleich, damit das Testset repräsentativ für die Gesamtverteilung ist.

Das Testset wird **ausschließlich für die finale Bewertung** verwendet – nie zur Modellauswahl oder Schwellenwertoptimierung. Würde man das Testset zwischendurch einsehen, flössen dort gewonnene Informationen in die Entwicklung ein und die Ergebnisse wären zu optimistisch.

**Ergebnis des Splits:**
- **Trainingsset:** 246.008 Beobachtungen, Ausfallrate 8,1 %
- **Testset:** 61.503 Beobachtungen, Ausfallrate 8,1 %

Die Ausfallrate ist in beiden Mengen identisch – die Stratifizierung hat funktioniert. Von den 61.503 Testfällen sind rund **4.965 echte Ausfälle**, die das Modell später erkennen soll.

In [2]:
X_train, X_test, y_train, y_test = prep.make_train_test_split(feat)
print("Train:", X_train.shape, "| Test:", X_test.shape)

21:43:08 | INFO    | Split: Train=246008 (Positivrate 0.081), Test=61503 (Positivrate 0.081)


Train: (246008, 159) | Test: (61503, 159)


## 2. Cross-Validation auf den Trainingsdaten

Bevor die Modelle endgültig trainiert werden, vergleichen wir sie per **5-facher Cross-Validation**: Das Trainingsset wird in 5 Teile aufgeteilt, jedes Modell wird 5-mal auf 4 Teilen trainiert und auf dem jeweils verbleibenden Teil bewertet. Das Ergebnis ist ein stabilerer Leistungsvergleich als ein einzelner Split.

Als Hauptmetrik dient der **PR-AUC** (Fläche unter der Precision-Recall-Kurve). Bei stark unausgewogenen Daten ist diese Metrik aussagekräftiger als Accuracy oder ROC-AUC, weil sie sich auf die seltene, aber relevante Positivklasse (Kreditausfall) konzentriert. Ein Modell, das immer „kein Ausfall" sagt, erreicht bei 8 % Ausfallrate einen PR-AUC von 0.08 – das ist das Zufallsniveau, das jedes echte Modell schlagen muss.

> **Laufzeit:** Die CV läuft auf einem stratifizierten Subsample (40.000 Zeilen statt 246.000), um die Rechenzeit beherrschbar zu halten. Die Modellrankings bleiben dabei statistisch stabil.

### Interpretation der CV-Ergebnisse

| Modell | PR-AUC (Mittelwert) | Stabilität (±Std) |
|---|---|---|
| LightGBM | **0.229** | ±0.013 |
| HistGradientBoosting | 0.226 | ±0.014 |
| Logistic Regression | 0.219 | ±0.012 |
| Random Forest | 0.198 | ±0.005 |
| Dummy (Baseline) | 0.081 | ±0.000 |

**Was fällt auf?**

- **Alle echten Modelle schlagen die Baseline** deutlich. Der Dummy mit PR-AUC 0.08 entspricht dem Zufallsniveau bei 8 % Ausfallrate – ein reales Modell muss das übertreffen.
- **Boosting-Modelle führen knapp**, aber der Vorsprung gegenüber Logistic Regression ist gering (~0.01). Das zeigt, dass ein einfaches lineares Modell hier bereits viel Vorhersagekraft liefert.
- **Random Forest liegt zurück**, ist aber am stabilsten (niedrigste Streuung ±0.005). Die anderen Modelle schwanken stärker zwischen den 5 Folds.
- **Absolute Werte im Kontext:** PR-AUC 0.23 klingt niedrig, ist aber bei 8 % Positivrate mit vielen fehlenden Werten ein realistisches Ergebnis für diesen Datensatz.

In [3]:
cv_results = tm.cross_validate_models(X_train, y_train, scoring="average_precision")
cv_df = pd.DataFrame([(k, v[0], v[1]) for k, v in cv_results.items()],
                     columns=["model", "cv_pr_auc_mean", "cv_pr_auc_std"]
            ).sort_values("cv_pr_auc_mean", ascending=False)
utils.save_table(cv_df.round(4), "cv_results", subdir="modeling")
cv_df

21:43:12 | INFO    | Demo-Subsampling: 246008 → 40000 Zeilen (stratifiziert)
21:43:12 | INFO    | Feature-Typen: 143 numerisch, 16 kategorial.
21:43:21 | INFO    | CV dummy          average_precision = 0.0807 +/- 0.0001
21:43:37 | INFO    | CV logreg         average_precision = 0.2185 +/- 0.0115
21:43:56 | INFO    | CV random_forest  average_precision = 0.1983 +/- 0.0046
21:44:23 | INFO    | CV hist_gbdt      average_precision = 0.2255 +/- 0.0137
21:44:38 | INFO    | CV lightgbm       average_precision = 0.2293 +/- 0.0127
21:44:38 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\modeling\cv_results.csv


,model,cv_pr_auc_mean,cv_pr_auc_std
4,lightgbm,0.229307,0.012683
3,hist_gbdt,0.225540,0.013652
1,logreg,0.218528,0.011541
2,random_forest,0.198312,0.004634
0,dummy,0.080725,0.000050


## 3. Modelle fitten & auf dem Testset vergleichen

**Das Vorgehen im Code:**
* **`fit_all_models`:** Alle Modelle werden nun final auf den *gesamten* Trainingsdaten (`X_train`, `y_train`) trainiert (gefittet). Sie lernen also anhand der maximal verfügbaren Informationsmenge.
* **`compare_models`:** Anschließend werden diese fertig trainierten Modelle **streng einmalig** auf das bisher völlig ungesehene Testset (`X_test`, `y_test`) losgelassen. Hier berechnen wir alle relevanten Metriken, um zu sehen, wie die Modelle in einem echten Einsatzszenario performen würden.
* **`.round(4)`:** Die finale Ergebnistabelle wird der besseren Lesbarkeit halber auf vier Nachkommastellen gerundet.

**Achtung: Die "Imbalance-Falle" (Genauigkeits-Paradoxon)**
Da wir es mit einem stark ungleichgewichteten Datensatz zu tun haben, ist bei der Interpretation der Tabelle höchste Vorsicht geboten:
* Ein schlechtes Modell könnte einfach pauschal *immer* die Mehrheitsklasse (z. B. "Kein Ausfall") vorhersagen. Bei 90 % normalen Fällen und 10 % Ausfällen hätte dieses Modell eine Accuracy von satten 90 %, obwohl es in Wahrheit völlig nutzlos ist.

In [4]:
fitted = tm.fit_all_models(X_train, y_train)
comparison = ev.compare_models(fitted, X_test, y_test)
comparison.round(4)

22:02:01 | INFO    | Feature-Typen: 143 numerisch, 16 kategorial.
22:02:13 | INFO    | Gefittet: dummy
22:02:46 | INFO    | Gefittet: logreg
22:03:29 | INFO    | Gefittet: random_forest
c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Das System kann die angegebene Datei nicht finden
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\si

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
0,hist_gbdt,0.5,0.9201,0.6161,0.0262,0.0502,0.7748,0.2685
1,lightgbm,0.5,0.7139,0.1765,0.6937,0.2813,0.7751,0.2684
2,logreg,0.5,0.6982,0.1677,0.6914,0.2700,0.7629,0.2463
3,random_forest,0.5,0.9192,0.4684,0.0075,0.0147,0.7398,0.2169
4,dummy,0.5,0.9193,0.0000,0.0000,0.0000,0.5000,0.0807


### Interpretation der Ergebnisse

**Achtung: Die "Imbalance-Falle" (Genauigkeits-Paradoxon)**
Da wir es mit einem stark ungleichgewichteten Datensatz zu tun haben, ist bei der Interpretation der Tabelle höchste Vorsicht geboten. Ein schlechtes Modell könnte einfach pauschal immer die Mehrheitsklasse (z. B. "Kein Ausfall") vorhersagen.

* **Der Beweis durch das Dummy-Modell:** Der `dummy`-Klassifikator (sagt immer die Mehrheitsklasse vorher) erreicht eine scheinbar hervorragende *Accuracy* (Genauigkeit) von 91,93 %. Da er aber nie einen Ausfall vorhersagt, liegt sein *Recall* bei 0. Er erkennt keinen einzigen Kreditausfall und ist in der Praxis völlig nutzlos. Genau deshalb sind **Recall, F1 und PR-AUC** in unserem Fall die entscheidenden Größen.

**Wie schlagen sich die echten Modelle?**

* **Die Verlierer beim Standard-Threshold (Hist_GBDT & Random Forest):** Beide Modelle erzielen eine sehr hohe Genauigkeit (~92 %), die fast identisch mit dem Dummy-Modell ist. Ein Blick auf den *Recall* verrät das Problem: Der Random Forest findet nur 0,75 % der tatsächlichen Ausfälle, das Hist_GBDT-Modell nur 2,62 %. Bei einem festen Schwellenwert (Threshold) von 0.5 stufen diese Modelle fast alle Kunden als "sicher" ein.
* **Die Ausfall-Erkenner (LightGBM & Logistische Regression):** Diese Modelle haben eine deutlich niedrigere *Accuracy* (~70 % bis 71 %), sind aber für unseren Anwendungsfall weitaus besser geeignet. LightGBM erkennt knapp 69,4 % der tatsächlichen Ausfälle (*Recall* = 0.6937), LogReg liegt mit 69,1 % fast gleichauf.
* **Der Preis für den hohen Recall:** Die *Precision* bei LightGBM und LogReg ist mit ca. 17 % sehr niedrig. Das bedeutet: Wenn das Modell einen Kreditausfall vorhersagt, tritt dieser nur in etwa 17 % der Fälle wirklich ein. Es gibt also viele falsche Alarme (False Positives). Im Kreditwesen ist es jedoch oft günstiger, einen Kunden fälschlicherweise genauer zu prüfen, als einen tatsächlichen Kreditausfall (False Negative) zu übersehen.
* **Gesamtleistung (ROC-AUC & PR-AUC):** Die unabhängigen Metriken *ROC-AUC* und *PR-AUC*, die nicht vom festen Threshold (0.5) abhängen, zeigen, dass LightGBM und Hist_GBDT das höchste Potenzial besitzen, die Klassen sauber voneinander zu trennen (*ROC-AUC* von ca. 0.775 und *PR-AUC* von ca. 0.268). Um die schwache Recall-Performance von Hist_GBDT zu beheben, müsste man den Klassifizierungs-Schwellenwert (Threshold) aktiv anpassen.

## 4. ROC- und PR-Kurven

Die Kurven zeigen die Modellleistung **über alle möglichen Schwellenwerte** hinweg – unabhängig vom Default-Wert 0.5. So lassen sich Modelle fair vergleichen, ohne dass die Wahl eines bestimmten Schwellenwerts das Bild verzerrt.

- **ROC-Kurve** (links): Trägt den Anteil korrekt erkannter Ausfälle (True-Positive-Rate) gegen den Anteil fälschlich gemeldeter Nicht-Ausfälle (False-Positive-Rate) auf. Je weiter oben links die Kurve liegt, desto besser. Die gestrichelte Diagonale entspricht reinem Raten (ROC-AUC = 0.5).
- **PR-Kurve** (rechts): Trägt Precision (Genauigkeit der Vorhersagen) gegen Recall (Anteil erkannter Ausfälle) auf. Die gestrichelte Linie markiert die Positivrate (~8 %) – das ist das Niveau, das ein naives Modell erreicht. Werte darüber zeigen echte Vorhersagekraft.

### Interpretation der Kurven

**ROC-AUC:**
- `hist_gbdt` und `lightgbm`: **0.775** – das Modell ordnet echte Ausfälle mit 77,5 % Wahrscheinlichkeit korrekt vor Nicht-Ausfällen ein.
- `logreg`: 0.763 – nur minimal schlechter, trotz viel einfacherer Modellstruktur.
- `random_forest`: 0.740 – schwächer im Ranking.
- Zum Vergleich: Zufall = 0.5, perfektes Modell = 1.0.

**PR-AUC:**
- Beste Modelle erreichen ~0.27, weit über dem Zufallsniveau 0.08 (= Positivrate). Das belegt echte Vorhersagekraft.
- `logreg` (0.246) liegt nur 0.02 hinter den Boosting-Modellen – für ein interpretierbareres Modell ein sehr gutes Ergebnis.
- In der PR-Kurve sieht man deutlich, dass hoher Recall nur mit stark sinkender Precision erkauft wird: Um alle Ausfälle zu finden, müsste man viele Nicht-Ausfälle fälschlich melden.

In [5]:
fig = ev.plot_roc_pr_curves(fitted, X_test, y_test); plt.show()

c:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
22:08:58 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\modeling\roc_pr_curves.png
C:\Users\annis\AppData\Local\Temp\ipykernel_25544\4061997854.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = ev.plot_roc_pr_curves(fitted, X_test, y_test); plt.show()


## 5. Der Schwellenwert-Trade-off

Das Modell gibt für jeden Kreditantrag eine **Wahrscheinlichkeit** aus (0–1). Erst durch die Wahl eines **Schwellenwerts** wird daraus eine binäre Entscheidung: „Ausfall vorhergesagt" wenn Wahrscheinlichkeit ≥ Schwelle, sonst „kein Ausfall".

| Schwelle | Effekt |
|---|---|
| **Hoch (z. B. 0.5)** | Modell ist vorsichtig, meldet nur sehr sichere Fälle. → Wenig Fehlalarme, aber viele echte Ausfälle werden übersehen (hohe False Negatives). |
| **Niedrig (z. B. 0.15)** | Modell meldet viel mehr Fälle als Ausfall. → Mehr echte Ausfälle erkannt, aber auch mehr kreditwürdige Personen fälschlich abgelehnt (hohe False Positives). |

**Das ist keine rein technische Entscheidung.** Welcher Fehler schwerer wiegt, hängt von Geschäftsstrategie und ethischen Abwägungen ab:
- Ein **nicht erkannter Ausfall** (False Negative) bedeutet einen bewilligten Kredit, der nicht zurückgezahlt wird → finanzieller Verlust für die Bank.
- Eine **fälschliche Ablehnung** (False Positive) bedeutet, dass eine kreditwürdige Person keinen Kredit bekommt → entgangener Ertrag und mögliche Benachteiligung der Person.

### Interpretation der Threshold-Analyse (hist_gbdt, Testset: 4.965 echte Ausfälle)

| Schwelle | Precision | Recall | F1 | Flagged als Ausfall |
|---|---|---|---|---|
| 0.50 | 61.6 % | 2.6 % | 5 % | 211 |
| 0.30 | 41.2 % | 16.2 % | 23 % | 1.948 |
| 0.20 | 30.9 % | 31.9 % | **31 %** | 5.125 |
| **0.15** | 25.7 % | **44.7 %** | **33 %** | 8.647 |
| 0.10 | 19.9 % | 61.9 % | 30 % | 15.415 |

**Was bedeuten diese Zahlen konkret?**

- **Schwelle 0.50:** Nur 211 Anträge werden als risikoreich markiert, davon ~130 echte Ausfälle. Die restlichen **4.835 echten Ausfälle** werden als sicher eingestuft und der Kredit bewilligt.
- **Schwelle 0.15:** 8.647 Anträge werden geflaggt. Davon sind ~4.457 echte Ausfälle erkannt (45 %). Gleichzeitig werden ~4.190 kreditwürdige Personen fälschlich abgelehnt.
- **Schwelle 0.10:** 61.9 % der Ausfälle erkannt – aber über 12.000 falsche Alarme.

**Empfehlung:** Ein Schwellenwert zwischen **0.15 und 0.20** balanciert Recall und Precision am besten (höchstes F1 ≈ 0.33). Die finale Wahl bleibt eine geschäftliche Entscheidung.

In [6]:
best_name = comparison[comparison.model != "dummy"].iloc[0]["model"]
print("Bestes Modell (nach PR-AUC):", best_name)
thr = ev.threshold_analysis(fitted[best_name], X_test, y_test, best_name)
thr

Bestes Modell (nach PR-AUC): hist_gbdt


22:09:22 | INFO    | Threshold-Analyse (hist_gbdt):
 threshold  precision  recall    f1  predicted_positive
      0.50      0.616   0.026 0.050                 211
      0.30      0.412   0.162 0.232                1948
      0.20      0.309   0.319 0.314                5125
      0.15      0.257   0.447 0.326                8647
      0.10      0.199   0.619 0.302               15415
22:09:22 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\modeling\threshold_analysis_hist_gbdt.csv


,threshold,precision,recall,f1,predicted_positive
0,0.50,0.616114,0.026183,0.050232,211
1,0.30,0.411704,0.161531,0.232027,1948
2,0.20,0.309073,0.319033,0.313974,5125
3,0.15,0.256621,0.446928,0.326036,8647
4,0.10,0.199416,0.619134,0.301668,15415


## 6. Confusion Matrix & Classification Report

Die Confusion Matrix zeigt auf einen Blick, wie die Vorhersagen des besten Modells (`hist_gbdt`, Schwelle 0.5) auf die vier möglichen Fälle verteilt sind:

| | **Vorhergesagt: kein Ausfall** | **Vorhergesagt: Ausfall** |
|---|---|---|
| **Tatsächlich: kein Ausfall** | True Negative (TN) – korrekt abgelehnt ✓ | False Positive (FP) – fälschlich als Ausfall markiert |
| **Tatsächlich: Ausfall** | False Negative (FN) – übersehener Ausfall | True Positive (TP) – korrekt erkannt ✓ |

Im Kreditkontext ist die **FN-Zelle** besonders kritisch: Jeder False Negative ist ein bewilligter Kredit, der später ausfällt.

### Interpretation (hist_gbdt, Schwelle 0.5)

Das Testset enthält **56.538 Nicht-Ausfälle** und **4.965 echte Ausfälle**.

| Kategorie | Anzahl | Bedeutung |
|---|---|---|
| True Negative (TN) | ~56.408 | Kreditwürdige Personen korrekt als sicher eingestuft ✓ |
| False Positive (FP) | ~81 | Kreditwürdige Personen fälschlich abgelehnt |
| **False Negative (FN)** | **~4.835** | **Echte Ausfälle nicht erkannt – Kredit fälschlich bewilligt|
| True Positive (TP) | ~130 | Ausfälle korrekt erkannt ✓ |

Das Modell erkennt bei Standardschwelle 0.5 fast keine Ausfälle (**Recall = 3 %**). Die hohe Accuracy (92 %) ist irreführend – sie entsteht fast ausschließlich durch die korrekte Klassifikation der Mehrheitsklasse.

Ein Schwellenwert von 0.15 würde statt 130 rund **4.457 Ausfälle** erkennen – auf Kosten von mehr False Positives (vgl. Abschnitt 5).

In [7]:
fig, cm = ev.plot_confusion(fitted[best_name], X_test, y_test, best_name); plt.show()
print(ev.full_classification_report(fitted[best_name], X_test, y_test, best_name))

22:09:28 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\modeling\confusion_hist_gbdt.png
C:\Users\annis\AppData\Local\Temp\ipykernel_25544\1856836977.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig, cm = ev.plot_confusion(fitted[best_name], X_test, y_test, best_name); plt.show()


                  precision    recall  f1-score   support

kein Ausfall (0)       0.92      1.00      0.96     56538
     Ausfall (1)       0.62      0.03      0.05      4965

        accuracy                           0.92     61503
       macro avg       0.77      0.51      0.50     61503
    weighted avg       0.90      0.92      0.88     61503



## 7. Kalibrierung des besten Modells

Ein Modell ist **gut kalibriert**, wenn seine vorhergesagte Wahrscheinlichkeit der tatsächlichen Ausfallrate entspricht. Wenn das Modell z. B. für 100 Anträge eine Ausfallwahrscheinlichkeit von 20 % ausgibt, sollten darunter tatsächlich ~20 Ausfälle sein.

Das ist besonders relevant, wenn die **Wahrscheinlichkeit selbst** weiterverwendet wird – z. B. zur Berechnung des erwarteten Verlusts (Expected Loss = Ausfallwahrscheinlichkeit × Verlustquote × Kreditbetrag).

> **Wichtig:** Kalibrierung und Trennschärfe sind unabhängig voneinander. Ein Modell kann Ausfälle gut nach Risiko **sortieren** (hoher ROC-AUC) und gleichzeitig die Wahrscheinlichkeiten systematisch über- oder unterschätzen (schlechte Kalibrierung).

### Interpretation der Kalibrierungskurve

- **Punkte nahe der Diagonalen:** gut kalibriert – vorhergesagte Wahrscheinlichkeit ≈ beobachtete Ausfallrate.
- **Punkte über der Diagonalen:** Modell unterschätzt die Ausfallwahrscheinlichkeit (zu optimistisch).
- **Punkte unter der Diagonalen:** Modell überschätzt die Ausfallwahrscheinlichkeit (zu pessimistisch).

Gradient-Boosting-Modelle neigen dazu, Wahrscheinlichkeiten nahe 0 und 1 zu überschätzen (S-förmige Abweichung von der Diagonalen). Wenn die ausgegebene Wahrscheinlichkeit direkt in Geschäftsentscheidungen einfließt, sollte eine **nachgelagerte Kalibrierung** in Betracht gezogen werden.

### Analyse `hist_gbdt`

Ein Blick auf unsere aktuelle Kalibrierungskurve zeigt zwei wesentliche Erkenntnisse:

* **Kalibrierung:** Die blaue Linie unseres `hist_gbdt`-Modells liegt beinahe perfekt auf der gestrichelten Diagonalen. Entgegen der typischen Schwäche von Gradient-Boosting-Modellen ist dieses Modell also **sehr gut kalibriert**. Wenn das Modell ein Risiko von 10 % oder 20 % prognostiziert, deckt sich dies hochgradig präzise mit der Realität.
* **Eingeschränkter Vorhersagebereich (bis ~30 %):** Es fällt auf, dass die Kurve bei einer mittleren vorhergesagten Wahrscheinlichkeit von knapp 0.3 (30 %) endet. Das Modell sagt für keinen Kunden Wahrscheinlichkeiten von beispielsweise 80 % oder 90 % voraus. Dies ist eine direkte Folge des starken **Klassenungleichgewichts** (die Basis-Ausfallrate im Datensatz liegt bei nur 8,1 %). Das Modell weiß, dass ein absolut sicherer Ausfall (nahe 100 %) statistisch extrem unwahrscheinlich ist, bewertet die vorhandenen Risiken im unteren bis mittleren Segment dafür aber absolut exakt.

In [8]:
fig = ev.plot_calibration(fitted[best_name], X_test, y_test, best_name); plt.show()

22:09:36 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\modeling\calibration_hist_gbdt.png
C:\Users\annis\AppData\Local\Temp\ipykernel_25544\4238717784.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = ev.plot_calibration(fitted[best_name], X_test, y_test, best_name); plt.show()


## 8. Hyperparameter-Tuning & finales Modell

Das beste Modell (`hist_gbdt`) wird mit einer **Grid Search** (3-facher CV, ~24 Parameterkombinationen) weiter optimiert. Der Suchraum ist bewusst klein gehalten – ein sehr großer Raum erhöht die Gefahr, zufällige Schwankungen im Validierungsset zu überfitten.

Das Tuning läuft auf dem stratifizierten Subsample (40.000 Zeilen), die gefundenen Parameter werden anschließend auf dem **vollen Trainingsset** verwendet. Das fertige Modell wird als eine einzige Datei gespeichert (`best_model.joblib`), die **Preprocessing + Klassifikator** in einer Pipeline enthält.
### Interpretation der Tuning-Ergebnisse

**Beste Parameter:** `learning_rate = 0.03`, `max_iter = 200`, `max_leaf_nodes = 31`

| | Vor Tuning | Nach Tuning |
|---|---|---|
| CV PR-AUC | 0.2255 | **0.2259** |
| Test PR-AUC | 0.2685 | 0.2450 |
| Test Recall (Schwelle 0.5) | 2.6 % | 1.2 % |

**Das Tuning bringt kaum Verbesserung.** Die CV PR-AUC steigt minimal um 0.0004 – das liegt innerhalb der normalen Schwankungsbreite. Das ist typisch für Gradient-Boosting auf tabellarischen Daten: die Standardparameter sind bereits gut gewählt, und der begrenzte Suchraum findet keine wesentlich besseren Einstellungen.

Der leichte Rückgang des Test-PR-AUC (0.269 → 0.245) deutet darauf hin, dass die auf dem Subsample (40.000 Zeilen) gefundenen Parameter nicht perfekt auf den vollen Datensatz übertragen. Für Produktionszwecke wäre ein Tuning auf dem vollständigen Trainingsset sinnvoller.

In [9]:
best_est, best_params, best_score = tm.tune_best_model(X_train, y_train)
print("Beste Parameter:", best_params, "| CV PR-AUC:", round(best_score, 4))
tm.save_model(best_est, "best_model")
final_metrics = ev.evaluate_model(best_est, X_test, y_test, "best_model_tuned")
final_metrics

22:09:40 | INFO    | Demo-Subsampling: 246008 → 40000 Zeilen (stratifiziert)
22:09:40 | INFO    | Feature-Typen: 143 numerisch, 16 kategorial.
22:11:20 | INFO    | Bestes average_precision = 0.2259 bei {'classifier__learning_rate': 0.03, 'classifier__max_iter': 200, 'classifier__max_leaf_nodes': 31}
22:11:21 | INFO    | Modell gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\models\best_model.joblib


Beste Parameter: {'classifier__learning_rate': 0.03, 'classifier__max_iter': 200, 'classifier__max_leaf_nodes': 31} | CV PR-AUC: 0.2259


22:11:23 | INFO    | best_model_tuned | Acc=0.920 P=0.600 R=0.012 F1=0.024 ROC=0.760 PR=0.245


{'model': 'best_model_tuned',
 'threshold': 0.5,
 'accuracy': 0.9195974180121295,
 'precision': 0.6,
 'recall': 0.012084592145015106,
 'f1': 0.023692003948667325,
 'roc_auc': 0.7595008741547407,
 'pr_auc': 0.24500016584226572}

## 9. Fazit & Zusammenfassung

### Modellleistung im Überblick (Testset)

| Modell | PR-AUC | ROC-AUC | Recall bei Schwelle 0.5 | Hinweis |
|---|---|---|---|---|
| hist_gbdt | **0.269** | 0.775 | 2.6 % | Bestes Ranking, sehr konservativ bei 0.5 |
| lightgbm | 0.268 | 0.775 | 69.4 % | Gleichwertig, höherer Recall bei 0.5 |
| logreg | 0.246 | 0.763 | 69.1 % | Beste Wahl für Erklärbarkeit |
| random_forest | 0.217 | 0.740 | 0.8 % | Schwächstes echtes Modell |
| dummy | 0.081 | 0.500 | 0.0 % | Baseline – nicht verwendbar |

### Zentrale Erkenntnisse

**1. Accuracy ist bei unausgewogenen Daten wertlos als Hauptmetrik.**  
Dummy und Random Forest erreichen ~92 % Accuracy, ohne einen einzigen Ausfall zu erkennen. PR-AUC, Recall und F1 sind die relevanten Größen.

**2. Der Schwellenwert 0.5 ist für Baummodelle ungeeignet.**  
`hist_gbdt` erkennt bei Schwelle 0.5 nur ~130 von 4.965 Ausfällen. Bei 0.15 steigt die Erkennungsrate auf ~45 % – allerdings mit entsprechend mehr Fehlalarmen. Die Wahl der Schwelle ist eine **geschäftliche Entscheidung**, keine technische Konstante.

**3. Logistic Regression ist konkurrenzfähig.**  
Mit PR-AUC 0.246 liegt sie nur knapp hinter den Boosting-Modellen. In einem regulierten Kreditumfeld (z. B. DSGVO-Auskunftspflicht, Begründungspflicht bei Ablehnung) wäre sie aufgrund ihrer **Interpretierbarkeit** die bevorzugte Wahl.

**4. Das absolute Leistungsniveau ist eingeschränkt**  
PR-AUC ~0.27 zeigt, dass echte Kreditausfälle schwer vorherzusagen sind. Viele relevante Faktoren (Gesundheit, Jobverlust, unvorhergesehene Ausgaben) sind im Datensatz nicht erfasst. Ein höherer Wert wäre eher ein Warnsignal für Leakage als ein reales Leistungsmerkmal.

**5. Kalibrierung prüfen, wenn Wahrscheinlichkeiten genutzt werden.**  
Soll die Ausfallwahrscheinlichkeit direkt (z. B. für Expected-Loss-Berechnungen) verwendet werden, sollte das Modell kalibriert werden.

**Nächstes Notebook (05):** Unüberwachte Segmentierung – Clustering der Kreditnehmer nach Risikoprofil.